# Python Quiz Study Guide
## Deep Dive: Pandas Operations & Function Anatomy
**Updated with Columns Deep Dive & Function Indexing/Calling**

**DATASCI 151 — Lectures 14, 17, 18a, 18b**

---

### Table of Contents

**Part 1 — Pandas Operations Deep Dive**
1. `pd.read_csv()`
2. `.head()`
3. `.columns` / `.columns.values`
4. ⭐ DEEP DIVE: Everything About Columns
5. `.iloc` — Integer-Location Based Indexing
6. `.sort_values()`
7. `.query()` — Filtering Rows
8. `.shape` and `len()`
9. Adding New Columns
10. `.apply()`
11. `pd.unique()`
12. `pd.merge()`
13. `os.listdir()`
14. Plotting

**Part 2 — Anatomy of a Function**
15. Structure of Every Function
16. Function Types You Must Know
17. ⭐ DEEP DIVE: Function Indexing & Calling
18. Quick Reference Cheat Sheet

---
# Part 1 — Pandas Operations Deep Dive
---

# 1. `pd.read_csv()` — Reading Data

**What it does:** Reads a CSV file and creates a DataFrame (a table with named columns and numbered rows).

- The argument is a string with the file path relative to your notebook
- If the file is in a subfolder, include the path: `"quiz_data/features.csv"`
- The first row of the CSV automatically becomes the column names
- Always store the result in a variable

In [ ]:
import pandas as pd

# df = pd.read_csv("quiz_data/features.csv")
# print(type(df))  # <class 'pandas.core.frame.DataFrame'>

---
# 2. `.head()` — Preview Rows

**What it does:** Returns the first N rows (default 5). It's a **method**, so use parentheses.

In [ ]:
# df.head()      # first 5 rows
# df.head(10)    # first 10 rows

---
# 3. `.columns` / `.columns.values` — Column Names

**What it does:** `.columns` is an **ATTRIBUTE** (no parentheses!). Returns a Pandas Index. Chain `.values` to get a NumPy array of strings.

In [ ]:
# df.columns           # Pandas Index object
# df.columns.values    # NumPy array of strings

> ⚠️ Writing `df.columns()` with parentheses will cause an error. It's an attribute, not a method.

---
---
# 4. ⭐ DEEP DIVE: Everything About Columns
---

## What Is a Column?

A DataFrame is a table. Each column has a **name** (a string) and holds one type of data. When you read a CSV with `pd.read_csv()`, the first row of the file automatically becomes the column names. Every other row becomes a data row.

## Getting Column Names

### Option A: `df.columns`
This is an **attribute**, not a method. No parentheses! It returns a Pandas Index object.

In [ ]:
# car_cols = df.columns
# print(car_cols)
# Index(['mpg', 'cylinders', 'displacement', 'horsepower',
#        'weight', 'acceleration', 'vehicle id'], dtype='object')

### Option B: `df.columns.values`
Chain `.values` to get a plain NumPy array of strings. This is often easier to work with because you can index into it by position.

In [ ]:
# col_names = df.columns.values
# print(col_names)    # ['mpg' 'cylinders' 'displacement' ...]
#
# Index into it:
# print(col_names[0])   # 'mpg'
# print(col_names[4])   # 'weight'

> ✅ **Tip:** This is useful when you want to grab a column by its position number rather than typing the name.

## Extracting a Single Column

Use a single string inside square brackets. The result is a **Pandas Series** — a 1D labeled array, like one column plucked out of the spreadsheet.

In [ ]:
# mpg = df["mpg"]
# print(type(mpg))    # <class 'pandas.core.series.Series'>

A Series behaves very similarly to a NumPy array. You can do math on it, plot it, call `.apply()` on it, compare it, etc.

### Using column position instead of name

In [ ]:
# These two are equivalent:
# mpg = df["mpg"]
# mpg = df[df.columns.values[0]]    # column 0 is "mpg"

## Extracting Multiple Columns

Pass a **LIST** of strings inside the square brackets. The result is a **DataFrame** (a mini-table with just those columns).

In [ ]:
# subset = df[["weight", "mpg"]]    # note DOUBLE brackets!

### What the double brackets actually mean

The double brackets are **NOT** special syntax. Here's what's really happening:

In [ ]:
# Step 1: Create a list
cols_I_want = ["weight", "mpg"]

# Step 2: Pass that list into the brackets
# subset = df[cols_I_want]

# This is IDENTICAL to:
# subset = df[["weight", "mpg"]]

The outer brackets are the DataFrame indexing operator. The inner brackets define a Python list. When you write them inline, they just happen to sit next to each other, creating the "double bracket" look.

## The #1 Source of Column Mistakes

Understanding the difference between single and double brackets is **critical**:

| Code | What You Get | Type Returned |
|------|-------------|---------------|
| `df["mpg"]` | One column of data | **Series** |
| `df[["mpg"]]` | One column, but wrapped in a table | **DataFrame** |
| `df[["mpg", "weight"]]` | Two columns | **DataFrame** |
| `df["mpg", "weight"]` | ❌ **ERROR — crashes!** | — |

**Row 3 vs Row 4** is the key distinction. `df["mpg", "weight"]` fails because you're passing two separate strings, not a list. You need the inner brackets to make it a list first.

**Row 1 vs Row 2** is subtle but important: `df[["mpg"]]` (double brackets with one name) gives you a DataFrame with one column, while `df["mpg"]` (single brackets) gives you a Series. Same data, different types.

## Extracting Columns with `.iloc`

Instead of using column names, you can grab columns by their **position number** using `.iloc`. The colon `:` in the row position means "all rows."

| Code | What It Grabs | Returns |
|------|--------------|----------|
| `df.iloc[:, 0]` | All rows of column 0 (mpg) | Series |
| `df.iloc[:, 3]` | All rows of column 3 (horsepower) | Series |
| `df.iloc[:, [0, 4, 6]]` | Columns 0, 4, 6 — all rows | DataFrame |
| `df.iloc[:, 2:5]` | Columns 2, 3, 4 — all rows | DataFrame |

> ✅ **Tip:** Use column names (`df["mpg"]`) when you know the name. Use `.iloc[:, 3]` when you need to select by position.

## Column Names with Spaces or Special Characters

For regular bracket extraction, spaces and special characters are **fine**:

In [ ]:
# df["Height(in cm)"]      # works perfectly
# df["miles per gallon"]   # also fine

But inside `.query()`, Python tries to parse the string as code. Spaces and parentheses confuse it. The fix is **backticks**:

In [ ]:
# This FAILS:
# df.query("Height(in cm) > 175")

# This WORKS:
# df.query("`Height(in cm)` > 175")

# Backtick is the key to the left of 1 on your keyboard

> ⚠️ This comes up directly on **Practice Quiz Q4**! The FIFA dataset's height column is `Height(in cm)`, which needs backticks in `.query()`.

## Putting It All Together: Column Workflow

Here's the typical sequence when working with columns on a quiz problem:

In [ ]:
# 1. Read the data
# df = pd.read_csv("quiz_data/fifa23_players_basic.csv")

# 2. Check what columns exist
# print(df.columns.values)

# 3. Extract a single column (Series)
# overall = df["Overall"]

# 4. Extract multiple columns (DataFrame)
# subset = df[["Overall", "Height(in cm)", "Weight(in kg)"]]

# 5. Filter rows using a column with special characters
# tall = df.query("`Height(in cm)` > 175")

# 6. Add a new column
# df["is_tall"] = df["`Height(in cm)`"] > 175

# 7. Apply a function to a column
# df["classification"] = df["Overall"].apply(my_function)

---
---
# Part 1 (continued): More Pandas Operations
---

# 5. `.iloc` — Integer-Location Based Indexing

Selects rows and/or columns by integer position (0-based). Uses **SQUARE BRACKETS**, not parentheses.

| Selection Type | Example | What It Grabs |
|---------------|---------|---------------|
| Single value | `df.iloc[0, 3]` | One cell: row 0, col 3 |
| Entire row | `df.iloc[0, :]` | All columns of row 0 (Series) |
| Entire column | `df.iloc[:, 0]` | All rows of column 0 (Series) |
| List of rows | `df.iloc[[0,4,396], :]` | Rows 0, 4, 396 — all cols |
| Slice | `df.iloc[0:5, :]` | Rows 0–4 (stop excluded!) |
| Negative index | `df.iloc[-3:, :]` | Last 3 rows |
| Rows + Cols | `df.iloc[[0,4], [0,4,6]]` | Specific rows AND cols |

> ✅ Slicing follows `start:stop` (stop excluded), just like `range()`. `df.iloc[0:5, :]` gives rows 0,1,2,3,4.

---
# 6. `.sort_values()` — Sorting

Returns a **new** DataFrame with rows sorted by one column. Does NOT modify the original.

In [ ]:
# sorted_df = df.sort_values(by="mpg")                  # ascending
# sorted_df = df.sort_values(by="mpg", ascending=False)  # descending

# Powerful combo: 5 lightest cars
# df.sort_values(by="weight").iloc[:5, :]

---
# 7. `.query()` — Filtering Rows

Returns a new DataFrame with only rows where the condition is True.

### Basic

In [ ]:
# df.query("mpg >= 25")
# df.query("cylinders == 8")

### Combining Conditions

In [ ]:
# df.query("(acceleration >= 12) and (acceleration < 18)")
# df.query("(cylinders == 4) or (cylinders == 8)")

### Using Variables with `@`

In [ ]:
# threshold = 25
# df.query("mpg >= @threshold")

> ⚠️ Without `@`, Pandas looks for a **COLUMN** named `threshold` instead of your variable.

---
# 8. `.shape` and `len()` — Dimensions

In [ ]:
# num_rows, num_cols = df.shape    # unpack tuple
# num_rows = df.shape[0]           # just rows (integer)
# num_rows = len(df)               # also just rows

> ⚠️ `df.shape` is a **TUPLE**. If a question asks for an integer, use `df.shape[0]` or `len(df)`.

---
# 9. Adding New Columns

In [ ]:
import numpy as np

# From math on existing columns
# df["BMI"] = wt_kg / ht_meters**2

# From random numbers
# df["random_var"] = np.random.uniform(0, 1, size=len(df))

# From np.random.choice
# df["GroupID"] = np.random.choice([1, 2, 3, 4], size=len(df))

---
# 10. `.apply()` — Apply Function to Every Element

Takes a function, runs it on every element in a Series, returns a new Series.

In [ ]:
def classify(value):
    if value > 85:
        return "Top Performer"
    else:
        return "Non Top-Performer"

# result = df["Overall"].apply(classify)

> ⚠️ Your function must work on a **SINGLE** value. `.apply()` handles the looping.

---
# 11. `pd.unique()` — Unique Values

In [ ]:
# categories = pd.unique(df["cylinders"])
# print(categories)   # array([8, 4, 6, 3, 5])

---
# 12. `pd.merge()` — Merging DataFrames

Combines two DataFrames by matching rows on a shared column.

| Argument | What It Means |
|----------|---------------|
| `left` | Primary DataFrame (row order preserved when how="left") |
| `right` | Secondary DataFrame (extra columns appended) |
| `how` | `"left"` = keep all left rows; `"right"` = keep all right rows |
| `on` | Column name both DataFrames share |

In [ ]:
# merged = pd.merge(left=df1, right=df2, how="left", on="raceId")

# Chaining merges:
# step1 = pd.merge(left=results, right=races, how="left", on="raceId")
# final = pd.merge(left=step1, right=circuits, how="left", on="circuitId")

> ⚠️ If both DFs share a column name (besides the `on` column), Pandas adds `_x` and `_y` suffixes. Subset columns before merging to avoid this.

---
# 13. `os.listdir()` — Listing Files

In [ ]:
import os

# folder = "quiz_data/country_emissions"
# file_names = os.listdir(folder)
#
# df_list = []
# for name in file_names:
#     full_path = folder + "/" + name
#     df = pd.read_csv(full_path)
#     df_list.append(df)

> ⚠️ `os.listdir` returns ONLY file names (e.g., `"germany_emission.csv"`). You must build the full path.

---
# 14. Plotting

### Histogram

In [ ]:
import matplotlib.pyplot as plt

# plt.hist(x=df["mpg"], alpha=0.5)
# plt.xlabel("MPG")
# plt.ylabel("Frequency")
# plt.title("MPG Distribution")
# plt.show()

### Scatter Plot

In [ ]:
# plt.scatter(x=df["weight"], y=df["acceleration"])
# plt.show()

### Loop Over Categories

In [ ]:
# for cat in pd.unique(df["cylinders"]):
#     sub = df.query("cylinders == @cat")
#     plt.scatter(x=sub["weight"], y=sub["acceleration"])
#
# plt.legend(labels=pd.unique(df["cylinders"]), title="Cylinders")
# plt.show()     # call ONCE at the end to overlay all plots

---
---
# Part 2 — Anatomy of a Function
---

# 15. The Structure of Every Function

Every Python function has the same four components:

In [ ]:
def function_name(param1, param2):    # HEADER
    # BODY: indented code
    result = param1 + param2
    return result                      # RETURN STATEMENT

| Component | What It Is | Example |
|-----------|------------|----------|
| `def` keyword | Tells Python you're defining a function | `def quadratic_formula(...)` |
| Function name | Descriptive, snake_case | `quadratic_formula` |
| Parameters | Input variables in parentheses | `(a, b, c)` |
| Body | Indented code that does the work | `discrim = b**2 - 4*a*c` |
| `return` | Sends value(s) back to caller | `return x_plus, x_minus` |

---
# 16. Function Types You Must Know

## Type A: Compute and Return a Value

The most common type. Takes inputs, does math or logic, returns a result.

### Example: Compound Interest

In [ ]:
def invest(P, r, n, t):
    V = P * (1 + r/n) ** (n*t)
    return round(V, 2)

print(invest(1000, 0.05, 12, 10))  # 1647.01

### Example: Alternate Quadratic Formula

In [ ]:
import numpy as np

def alt_quad_formula(a, b, c):
    discrim = np.sqrt(b**2 - 4*a*c)
    x1 = (2*c) / (-b - discrim)
    x2 = (2*c) / (-b + discrim)
    return x1, x2

root1, root2 = alt_quad_formula(3, -3, -6)
print("Root 1:", root1)   # -1.0
print("Root 2:", root2)   #  2.0

### Example: Boolean Return

In [ ]:
def check_voter_eligibility(age):
    return age >= 18    # returns True or False directly

print(check_voter_eligibility(17))   # False
print(check_voter_eligibility(21))   # True

## Type B: Perform an Action (No Return Value)

Does something visible (plotting, printing) but doesn't compute a value to hand back.

In [ ]:
import matplotlib.pyplot as plt

def create_histogram(vec_x, xlabel, title):
    plt.hist(x=vec_x)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.title(title)
    plt.show()

## Type C: Classify (for `.apply()`)

Takes a SINGLE value, returns a string/label. Designed to be passed into `.apply()`.

In [ ]:
def classify_player(Overall):
    if Overall > 85:
        return "Top Performer"
    else:
        return "Non Top-Performer"

# result = df["Overall"].apply(classify_player)

## Type D: Simulation / While Loop

Uses a while loop to simulate a random process, returns how many iterations it took.

In [ ]:
def run_consecutive(success_val):
    num_success = 0
    num_trials = 0
    while num_success < success_val:
        outcome = np.random.choice(["Success","Failure"], p=[0.8,0.2])
        num_trials = num_trials + 1
        if outcome == "Success":
            num_success = num_success + 1
        else:
            num_success = 0    # RESET on failure
    return num_trials

print(run_consecutive(3))
print(run_consecutive(7))

---
---
# 17. ⭐ DEEP DIVE: Function Indexing & Calling
---

## Calling a Function: What Actually Happens

When you call a function, Python does three things in order:
1. Takes the values you pass in and assigns them to the parameter names
2. Executes the body line by line
3. Hands back whatever comes after `return`

Let's trace through a concrete example:

In [ ]:
def invest(P, r, n, t):
    V = P * (1 + r/n) ** (n*t)
    return round(V, 2)

# CALLING the function:
amount = 1000
rate = 0.05
result = invest(amount, rate, 12, 10)
print(result)

Here is what Python does internally when it hits that last line:

| Step | What Happens | Values |
|------|-------------|--------|
| 1 | Maps arguments to parameters by POSITION | P=1000, r=0.05, n=12, t=10 |
| 2 | Executes: V = 1000 * (1 + 0.05/12)^(12*10) | V = 1647.009... |
| 3 | Executes: round(V, 2) | 1647.01 |
| 4 | Returns 1647.01 → assigned to 'result' | result = 1647.01 |

## Positional vs. Keyword Calling

When you call a function, you can pass arguments **by position** (order matters) or **by keyword name** (order doesn't matter). You can mix them, but positional arguments must come first.

| Call Style | Code | Valid? |
|-----------|------|--------|
| All positional | `invest(1000, 0.05, 12, 10)` | ✅ Yes |
| All keyword | `invest(P=1000, r=0.05, n=12, t=10)` | ✅ Yes |
| Keywords reordered | `invest(t=10, P=1000, n=12, r=0.05)` | ✅ Yes |
| Mixed (positional first) | `invest(1000, 0.05, n=12, t=10)` | ✅ Yes |
| Mixed (keyword first) | `invest(P=1000, 0.05, 12, 10)` | ❌ ERROR |

> ⚠️ Once you use a keyword argument, ALL remaining arguments must also be keyword. You cannot go back to positional.

## The Variable Names Do NOT Need to Match

This is one of the most confusing things about functions. The variable names you use when **calling** have NOTHING to do with the parameter names in the **definition**.

In [ ]:
def add(a, b):
    return a + b

# These ALL work and give the same result:
print(add(3, 7))           # a=3, b=7 → 10
print(add(a=3, b=7))       # a=3, b=7 → 10

x = 3
y = 7
print(add(x, y))           # a=3, b=7 → 10  (x maps to a, y maps to b)
print(add(y, x))           # a=7, b=3 → 10  (y maps to a, x maps to b!)
print(add(b=x, a=y))       # a=7, b=3 → 10  (keywords override position)

In the 4th call, `add(y, x)`, the variable `y` (which is 7) gets assigned to parameter `a`, and `x` (which is 3) gets assigned to `b`. For addition the result is the same either way, but for subtraction or division, **order matters!**

## What Happens When You Forget to Call

A common mistake: writing the function name **without parentheses**. This doesn't call the function — it just references the function object itself.

In [ ]:
def invest(P, r, n, t):
    V = P * (1 + r/n) ** (n*t)
    return round(V, 2)

result = invest            # NO parentheses = no call!
print(result)              # <function invest at 0x...>
print(type(result))        # <class 'function'>

result = invest(1000, 0.05, 12, 10)   # WITH parentheses = actual call
print(result)              # 1647.01

## Indexing Into Return Values

When a function returns multiple values, Python packs them into a **tuple**. You have two choices:

### Choice 1: Unpack into separate variables

In [ ]:
def alt_quad_formula(a, b, c):
    discrim = np.sqrt(b**2 - 4*a*c)
    x1 = (2*c) / (-b - discrim)
    x2 = (2*c) / (-b + discrim)
    return x1, x2

# Unpack: create two separate variables
root1, root2 = alt_quad_formula(3, -3, -6)
print(root1)     # -1.0
print(root2)     #  2.0

The number of variables on the left **MUST** match the number of values returned. If the function returns 2 values and you try to unpack into 3, you get a `ValueError`.

### Choice 2: Store as a tuple and index

In [ ]:
# Store everything in one variable (a tuple)
output = alt_quad_formula(3, -3, -6)
print(output)        # (-1.0, 2.0)
print(type(output))  # <class 'tuple'>

# Index into it
print(output[0])     # -1.0  (first value)
print(output[1])     #  2.0  (second value)

### Ignoring a return value with `_`

In [ ]:
# If you only care about one root:
_, root2 = alt_quad_formula(3, -3, -6)
print(root2)     # 2.0

# _ is a valid variable name, just a convention meaning "I don't care"

## Common Indexing Mistakes

| Mistake | What Happens | Fix |
|---------|-------------|-----|
| `x, y, z = func(a, b)` when func returns 2 values | `ValueError: not enough values to unpack` | Match the number: `x, y = func(a, b)` |
| `output[2]` when func returns 2 values | `IndexError: tuple index out of range` | Use `output[0]` or `output[1]` only |
| `result = func` without `()` | Stores the function itself, not its output | Add parentheses: `result = func()` |
| Using `result` as a number when func returned a tuple | `TypeError` on math operations | Unpack first, or index: `result[0]` |

## Calling Functions in Different Contexts

Functions don't just get called on their own line. Here are all the places you'll see function calls:

### Inside `print()`

In [ ]:
print(invest(1000, 0.05, 12, 10))     # prints 1647.01
print(check_voter_eligibility(17))     # prints False

The inner function runs first, then its return value is passed to `print()`.

### Inside `.apply()`

In [ ]:
# Pass the function NAME (no parentheses!) to .apply()
# result = df["Overall"].apply(classify_player)

# WRONG: calling the function yourself
# result = df["Overall"].apply(classify_player())  # ERROR!

> ⚠️ With `.apply()`, you pass the function name **WITHOUT** parentheses. Pandas calls it for you on each element. Adding `()` means YOU call it with no argument, which crashes.

### Inside other functions

In [ ]:
def F1(x, y, z):
    return x

def F2(x, y, z):
    return y**2 * z

def F(x, y, z):
    # F1 and F2 are CALLED here (with parentheses)
    return np.array([F1(x,y,z), F2(x,y,z)])

# When you call F(-1, 2, -3):
#   F internally calls F1(-1, 2, -3) → returns -1
#   F internally calls F2(-1, 2, -3) → returns -12
#   F returns np.array([-1, -12])
print(F(-1, 2, -3))

### Chaining: calling a method on a return value

In [ ]:
# pd.read_csv returns a DataFrame
# .query() is called on that DataFrame
# .shape is accessed on the result

# num_tall = pd.read_csv("file.csv").query("`Height(in cm)` > 175").shape[0]

# This is the same as:
# df = pd.read_csv("file.csv")
# df_tall = df.query("`Height(in cm)` > 175")
# num_tall = df_tall.shape[0]

> ✅ On a quiz, the multi-line version is safer and easier to debug.

## Using Functions Inside For Loops

A very common quiz pattern: define a function, then call it repeatedly inside a loop to build up a list.

In [ ]:
def check_voter_eligibility(age):
    return age >= 18

list_ages = [18, 29, 15, 32, 6]
list_eligible = []

for age in list_ages:
    result = check_voter_eligibility(age)    # call the function
    list_eligible.append(result)              # store the result

print(list_eligible)   # [True, True, False, True, False]

Each iteration:
1. The loop variable `age` takes on the next value from the list
2. We call the function with that value
3. We append the return value to our results list

After the loop, `list_eligible` has one entry per element in `list_ages`.

---
---
# 18. Quick Reference Cheat Sheet

| Operation | Code | Returns |
|-----------|------|----------|
| Read CSV | `pd.read_csv("file.csv")` | DataFrame |
| Preview rows | `df.head(n)` | DataFrame |
| Column names | `df.columns.values` | NumPy array |
| Single column | `df["col"]` | Series |
| Multiple columns | `df[["a","b"]]` | DataFrame |
| Row/col by position | `df.iloc[r, c]` | Value / Series / DF |
| Sort rows | `df.sort_values(by="col")` | DataFrame |
| Filter rows | `df.query("col > val")` | DataFrame |
| Filter with variable | `df.query("col > @var")` | DataFrame |
| Spaced column query | `` df.query("`col name` > 5") `` | DataFrame |
| Number of rows | `len(df)` or `df.shape[0]` | Integer |
| Shape (rows, cols) | `df.shape` | Tuple |
| Add column | `df["new"] = values` | Modifies df |
| Apply function | `series.apply(func)` | Series |
| Unique values | `pd.unique(df["col"])` | NumPy array |
| Merge two DFs | `pd.merge(left=, right=, how=, on=)` | DataFrame |
| List files | `os.listdir("folder")` | List of strings |
| Define function | `def func(a, b): ... return x` | — |
| Call function | `result = func(5, 10)` | Whatever func returns |
| Call with keywords | `result = func(a=5, b=10)` | Same as above |
| Unpack multiple returns | `x, y = func(a, b)` | Two variables |
| Index into return | `output = func(a, b); output[0]` | First value |
| Pass func to `.apply()` | `series.apply(func)` | No `()` on func! |

---
### Good luck on the quiz, Ian! 🎯